In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
data_path = Path("../dvc-stages-outputs/pv-italy/raw")

# Loading `local_features` table

In [3]:
df = pd.read_parquet(data_path / 'local_features.parquet')

In [4]:
grouped_df = df.groupby("site_id").agg(
    min_timestamp=pd.NamedAgg(column="timestamp", aggfunc="min"),
    max_timestamp=pd.NamedAgg(column="timestamp", aggfunc="max"),
    count=pd.NamedAgg(column="timestamp", aggfunc="count")
).reset_index()
grouped_df.sort_values(by=["max_timestamp", "count"], inplace=True)
grouped_df

,site_id,min_timestamp,max_timestamp,count
7,8,2012-01-01 02:00:00,2013-06-23 20:00:00,10260
4,5,2012-01-01 02:00:00,2014-05-06 20:00:00,12654
5,6,2012-01-01 02:00:00,2014-05-06 20:00:00,12654
6,7,2012-01-01 02:00:00,2014-05-06 20:00:00,12654
11,12,2012-01-01 02:00:00,2014-05-06 20:00:00,12654
10,11,2012-01-01 02:00:00,2014-05-06 20:00:00,13433
15,16,2012-01-01 02:00:00,2014-05-06 20:00:00,14934
16,17,2012-01-01 02:00:00,2014-05-06 20:00:00,14953
13,14,2012-01-01 02:00:00,2014-05-06 20:00:00,15428
2,3,2012-01-01 02:00:00,2014-05-06 20:00:00,16169


In [5]:
altieri_mask = (df["timestamp"] > pd.Timestamp("2012-01-01 20:00:00")) & (df['site_id'] != 8)
df = df[altieri_mask].drop(columns=["id", "model_id", "brand_id"])
df

,site_id,timestamp,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh
1,1,2012-01-08 03:00:00,-4,0,1012.96,4.22,0.78,1.34,296.0,0.31,0
2,1,2012-01-10 03:00:00,-4,0,1022.35,4.34,0.77,2.35,327.0,0.76,0
3,1,2012-01-07 07:00:00,-2,11,1008.70,9.08,0.58,0.49,350.0,0.31,74
4,1,2012-01-20 10:00:00,9,54,1019.14,5.12,0.67,5.42,212.0,0.31,240
5,1,2012-01-18 17:00:00,3,0,1029.61,1.91,0.85,2.88,353.0,0.59,0
...,...,...,...,...,...,...,...,...,...,...,...
265898,2,2012-08-31 04:00:00,-1,25,1013.28,1.48,0.68,12.74,227.0,0.00,5
265899,5,2012-12-05 06:00:00,0,8,999.52,5.80,0.77,3.93,246.0,0.35,18
265901,10,2013-06-23 08:00:00,-1,671,1013.33,3.33,0.50,15.81,337.0,0.41,529
265902,11,2013-11-07 07:00:00,-1,119,1015.01,1.05,0.76,9.80,263.0,0.31,331


In [6]:
reindexed_slices = []

# Process each site separately to handle different operational timeframes
for site_id, site_data in df.groupby('site_id'):
    min_date = site_data['timestamp'].min().date()
    max_date = site_data['timestamp'].max().date()
    
    # Generate complete sequence of calendar days
    all_days = pd.date_range(start=min_date, end=max_date, freq='D')
    
    # Generate expected 19 hours (02:00 to 20:00) per day
    expected_grid = []
    for day in all_days:
        for hour in range(2, 21):  # 2 to 20 inclusive
            expected_grid.append(pd.Timestamp(day).replace(hour=hour))
            
    grid_df = pd.DataFrame({'timestamp': expected_grid})
    grid_df['site_id'] = site_id
    
    # Left merge original data onto the complete grid (missing hours become NaN)
    merged = pd.merge(grid_df, site_data, on=['site_id', 'timestamp'], how='left')
    
    reindexed_slices.append(merged)
    
df_grid = pd.concat(reindexed_slices, ignore_index=True)
df_grid

,timestamp,site_id,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh
0,2012-01-02 02:00:00,1,0.0,0.0,0.00,2.24,0.82,-1.34,260.0,0.00,0.0
1,2012-01-02 03:00:00,1,0.0,0.0,1025.41,0.84,0.85,-1.46,206.0,0.11,0.0
2,2012-01-02 04:00:00,1,0.0,0.0,1025.41,3.58,0.84,-1.69,210.0,0.11,0.0
3,2012-01-02 05:00:00,1,0.0,0.0,1025.41,3.58,0.70,-1.08,220.0,0.11,0.0
4,2012-01-02 06:00:00,1,0.0,5.0,1025.16,3.58,0.82,-1.77,230.0,0.11,10.0
...,...,...,...,...,...,...,...,...,...,...,...
276483,2014-05-06 16:00:00,18,19.0,123.0,1020.20,1.49,0.56,9.41,76.0,0.31,25.0
276484,2014-05-06 17:00:00,18,18.0,23.0,1020.20,1.93,0.58,9.28,62.0,0.31,3.0
276485,2014-05-06 18:00:00,18,17.0,1.0,1019.91,0.89,0.63,8.82,89.0,0.31,0.0
276486,2014-05-06 19:00:00,18,16.0,1.0,1019.91,0.95,0.73,8.77,132.0,0.31,0.0


# Integrate with `azimuth_altitude` table

In [7]:
az_alt_df = pd.read_parquet(data_path / 'azimuth_altitude.parquet')
az_alt_df

,site_id,altitude,azimuth,timestamp
0,7,-57.24,-113.05,1900-01-01 00:00:00
1,7,-66.68,-134.84,1900-01-01 01:00:00
2,7,-71.92,-171.79,1900-01-01 02:00:00
3,7,-69.33,-146.46,1900-01-01 03:00:00
4,7,-60.96,-119.79,1900-01-01 04:00:00
...,...,...,...,...
157675,18,14.82,39.91,1900-12-30 17:00:00
157676,18,6.72,51.28,1900-12-30 18:00:00
157677,18,-2.70,61.36,1900-12-30 19:00:00
157678,18,-13.04,70.58,1900-12-30 20:00:00


In [8]:
dfs = []
for year in [2012, 2013, 2014]:
    new_rows = az_alt_df.copy()
    new_rows['timestamp'] = new_rows['timestamp'].apply(lambda x: x.replace(year=year))
    dfs.append(new_rows)
az_alt_df = pd.concat(dfs, ignore_index=True)
az_alt_df.sort_values(by=['site_id', 'timestamp'], inplace=True)

In [9]:
az_alt_df

,site_id,altitude,azimuth,timestamp
7662,1,-57.31,-113.06,2012-01-01 00:00:00
7663,1,-66.76,-134.90,2012-01-01 01:00:00
7664,1,-71.98,-172.00,2012-01-01 02:00:00
7665,1,-69.34,-146.19,2012-01-01 03:00:00
7666,1,-60.94,-119.60,2012-01-01 04:00:00
...,...,...,...,...
464419,18,-2.57,61.34,2014-12-31 19:00:00
464420,18,-12.91,70.56,2014-12-31 20:00:00
464421,18,-23.84,79.45,2014-12-31 21:00:00
464422,18,-35.10,88.64,2014-12-31 22:00:00


In [10]:
df_grid = df_grid.merge(az_alt_df, on=['site_id', 'timestamp'], how='left')
df_grid

,timestamp,site_id,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh,altitude,azimuth
0,2012-01-02 02:00:00,1,0.0,0.0,0.00,2.24,0.82,-1.34,260.0,0.00,0.0,-71.88,-171.68
1,2012-01-02 03:00:00,1,0.0,0.0,1025.41,0.84,0.85,-1.46,206.0,0.11,0.0,-69.31,-146.58
2,2012-01-02 04:00:00,1,0.0,0.0,1025.41,3.58,0.84,-1.69,210.0,0.11,0.0,-60.96,-119.88
3,2012-01-02 05:00:00,1,0.0,0.0,1025.41,3.58,0.70,-1.08,220.0,0.11,0.0,-50.42,-103.84
4,2012-01-02 06:00:00,1,0.0,5.0,1025.16,3.58,0.82,-1.77,230.0,0.11,10.0,-39.21,-92.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...
276483,2014-05-06 16:00:00,18,19.0,123.0,1020.20,1.49,0.56,9.41,76.0,0.31,25.0,55.23,54.37
276484,2014-05-06 17:00:00,18,18.0,23.0,1020.20,1.93,0.58,9.28,62.0,0.31,3.0,45.18,70.55
276485,2014-05-06 18:00:00,18,17.0,1.0,1019.91,0.89,0.63,8.82,89.0,0.31,0.0,34.17,82.61
276486,2014-05-06 19:00:00,18,16.0,1.0,1019.91,0.95,0.73,8.77,132.0,0.31,0.0,22.87,92.73


In [11]:
az_alt_nan_indexes = df_grid[df_grid["azimuth"].isnull() | df_grid["altitude"].isnull()].index
df_grid.loc[az_alt_nan_indexes]

,timestamp,site_id,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh,altitude,azimuth
1102,2012-02-29 02:00:00,1,-4.0,0.0,0.00,6.37,0.59,-1.51,277.0,0.00,0.0,NaN,NaN
1103,2012-02-29 03:00:00,1,-4.0,0.0,1015.31,4.98,0.56,-1.39,292.0,0.20,0.0,NaN,NaN
1104,2012-02-29 04:00:00,1,-4.0,0.0,1015.31,3.88,0.55,-0.96,294.0,0.31,0.0,NaN,NaN
1105,2012-02-29 05:00:00,1,-4.0,0.0,1015.31,4.69,0.58,0.04,287.0,0.31,0.0,NaN,NaN
1106,2012-02-29 06:00:00,1,0.0,14.0,1013.73,6.45,0.55,-0.25,300.0,0.47,44.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
261340,2012-02-29 16:00:00,18,6.0,19.0,1011.46,9.60,0.53,4.36,330.0,0.00,48.0,NaN,NaN
261341,2012-02-29 17:00:00,18,6.0,0.0,1011.46,8.51,0.56,4.37,330.0,0.00,0.0,NaN,NaN
261342,2012-02-29 18:00:00,18,5.0,0.0,1012.79,8.58,0.65,5.43,327.0,0.00,0.0,NaN,NaN
261343,2012-02-29 19:00:00,18,5.0,0.0,1012.79,7.05,0.70,5.81,337.0,0.00,0.0,NaN,NaN


# Imputation

## Imputation of leap days `azimuth` and `altitude` values

In [12]:
df_grid[(df_grid['timestamp'].dt.month == 2) & (df_grid['timestamp'].dt.day == 29)]["timestamp"].unique()

<DatetimeArray>
['2012-02-29 02:00:00', '2012-02-29 03:00:00', '2012-02-29 04:00:00',
 '2012-02-29 05:00:00', '2012-02-29 06:00:00', '2012-02-29 07:00:00',
 '2012-02-29 08:00:00', '2012-02-29 09:00:00', '2012-02-29 10:00:00',
 '2012-02-29 11:00:00', '2012-02-29 12:00:00', '2012-02-29 13:00:00',
 '2012-02-29 14:00:00', '2012-02-29 15:00:00', '2012-02-29 16:00:00',
 '2012-02-29 17:00:00', '2012-02-29 18:00:00', '2012-02-29 19:00:00',
 '2012-02-29 20:00:00']
Length: 19, dtype: datetime64[s]

In [13]:
df_grid.sort_values(by=["site_id", "timestamp"], inplace=True)

# impute remaining NaN values in azimuth and altitude using 
# azimuth and altitude values from the last day of february (28th) and the first day of march (1st)
for site_id, site_data in df_grid.groupby('site_id'):
    # Get the last day of February and the first day of March for each year
    feb_28 = site_data[(site_data['timestamp'].dt.month == 2) & (site_data['timestamp'].dt.day == 28)]
    mar_1 = site_data[(site_data['timestamp'].dt.month == 3) & (site_data['timestamp'].dt.day == 1)]
    
    # Create a mapping of timestamps to azimuth and altitude values
    az_alt_mapping = pd.concat([feb_28, mar_1]).set_index('timestamp')[['azimuth', 'altitude']].to_dict('index')
    
    # Impute NaN values for February 29th using the mapping
    for idx in site_data[(site_data['timestamp'].dt.month == 2) & (site_data['timestamp'].dt.day == 29)].index:
        timestamp = site_data.loc[idx, 'timestamp']
        year = timestamp.year
        
        # Get the corresponding February 28th and March 1st timestamps for the same year
        feb_28_timestamp = pd.Timestamp(year=year, month=2, day=28, hour=timestamp.hour)
        mar_1_timestamp = pd.Timestamp(year=year, month=3, day=1, hour=timestamp.hour)
        
        # Impute azimuth and altitude values as the average of February 28th and March 1st values
        if feb_28_timestamp in az_alt_mapping and mar_1_timestamp in az_alt_mapping:
            azimuth_avg = (az_alt_mapping[feb_28_timestamp]['azimuth'] + az_alt_mapping[mar_1_timestamp]['azimuth']) / 2
            altitude_avg = (az_alt_mapping[feb_28_timestamp]['altitude'] + az_alt_mapping[mar_1_timestamp]['altitude']) / 2
            
            df_grid.loc[idx, 'azimuth'] = azimuth_avg
            df_grid.loc[idx, 'altitude'] = altitude_avg

In [14]:
# Verification
missing_count = df_grid[["altitude", "azimuth"]].isna().sum()
print(f"Remaining missing values:\n{missing_count}")

Remaining missing values:
altitude    0
azimuth     0
dtype: int64


## With `external_ambiental_features` table

In [15]:
ambiental_nan_indexes = df_grid[df_grid.drop(columns=["kwh", "irradiance"]).isnull().any(axis=1)].index
df_grid.loc[ambiental_nan_indexes]

,timestamp,site_id,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh,altitude,azimuth
8683,2013-04-03 02:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-43.76,-176.32
8684,2013-04-03 03:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-42.44,-163.24
8685,2013-04-03 04:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37.44,-144.83
8686,2013-04-03 05:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-29.70,-129.46
8687,2013-04-03 06:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-20.18,-116.73
...,...,...,...,...,...,...,...,...,...,...,...,...,...
268921,2013-04-03 16:00:00,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.00,41.99
268922,2013-04-03 17:00:00,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.28,58.55
268923,2013-04-03 18:00:00,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.02,71.56
268924,2013-04-03 19:00:00,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.99,82.49


In [16]:
ext_amb_df = pd.read_parquet(data_path / 'external_ambiental_features.parquet')
ext_amb_df = ext_amb_df[['timestamp', 'site_id', 'temperature', 'pressure', 'windspeed', 'humidity', 'dewpoint', 'windbearing', 'cloudcover']]
ext_amb_df

,timestamp,site_id,temperature,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover
0,2012-01-01 03:00:00,1,5.03,1018.32,4.20,0.76,1.16,308.0,0.24
1,2012-01-01 04:00:00,1,4.76,1018.32,4.28,0.77,1.03,290.0,0.31
2,2012-01-01 05:00:00,1,4.76,1018.32,3.17,0.74,0.44,288.0,0.31
3,2012-01-01 06:00:00,1,4.21,1019.93,3.69,0.78,0.72,298.0,0.09
4,2012-01-01 07:00:00,1,4.34,1019.93,4.12,0.77,0.74,283.0,0.31
...,...,...,...,...,...,...,...,...,...
349267,2014-05-06 00:00:00,17,16.11,0.00,0.00,0.65,9.53,0.0,0.00
349268,2014-05-10 00:00:00,17,17.78,0.00,0.00,0.75,13.28,0.0,0.00
349269,2014-05-13 00:00:00,17,13.33,0.00,0.00,0.51,3.41,0.0,0.00
349270,2014-05-18 00:00:00,17,15.56,0.00,0.00,0.60,7.82,0.0,0.00


In [17]:
indexes_to_fill = df_grid.loc[ambiental_nan_indexes][['timestamp', 'site_id']]
fill_values = ext_amb_df.merge(indexes_to_fill, on=['timestamp', 'site_id'], how='inner')
fill_values.sort_values(by=['site_id', 'timestamp'], inplace=True)
fill_values[['timestamp', 'site_id', 'humidity', 'cloudcover']]

,timestamp,site_id,humidity,cloudcover
2785,2013-04-03 02:00:00,1,0.81,0.35
2786,2013-04-03 03:00:00,1,0.85,0.72
2787,2013-04-03 04:00:00,1,0.89,0.43
2788,2013-04-03 05:00:00,1,0.89,0.38
2789,2013-04-03 06:00:00,1,0.90,0.71
...,...,...,...,...
2932,2013-04-03 16:00:00,18,0.81,0.55
2933,2013-04-03 17:00:00,18,0.87,0.55
2934,2013-04-03 18:00:00,18,0.87,0.75
2935,2013-04-03 19:00:00,18,0.89,0.75


In [18]:
ambiental_cols_to_fill = list(fill_values.columns)
df_grid.loc[ambiental_nan_indexes, ambiental_cols_to_fill] = fill_values[ambiental_cols_to_fill].values
print("Number of remaining NaN values in the filled columns:")
print(df_grid.loc[ambiental_nan_indexes, ambiental_cols_to_fill].isnull().sum())

Number of remaining NaN values in the filled columns:
timestamp      0
site_id        0
temperature    0
pressure       0
windspeed      0
humidity       0
dewpoint       0
windbearing    0
cloudcover     0
dtype: int64


## With `external_irradiance_features` table

In [19]:
irradiance_nan_indexes = df_grid[df_grid["irradiance"].isnull()].index
na_df = df_grid.loc[irradiance_nan_indexes]
na_df.sort_values(by=["timestamp"], inplace=True)
na_df

,timestamp,site_id,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh,altitude,azimuth
150461,2012-08-04 02:00:00,11,21.84,NaN,1012.89,2.88,0.72,16.66,250.0,0.00,NaN,-31.76,-176.36
150462,2012-08-04 03:00:00,11,21.52,NaN,1012.23,2.79,0.73,16.42,305.0,0.00,NaN,-30.84,-166.89
150463,2012-08-04 04:00:00,11,20.96,NaN,1012.23,1.90,0.74,16.20,255.0,0.00,NaN,-26.78,-151.20
150464,2012-08-04 05:00:00,11,22.75,NaN,1012.23,0.32,0.69,16.77,264.0,0.00,NaN,-20.16,-137.33
150465,2012-08-04 06:00:00,11,26.39,NaN,1012.47,0.83,0.61,18.27,246.0,0.00,NaN,-11.65,-125.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
209037,2013-12-31 20:00:00,14,9.79,NaN,1017.21,1.84,1.00,9.79,319.0,0.82,NaN,-13.20,70.80
241565,2013-12-31 20:00:00,16,-1.91,NaN,0.00,2.24,0.93,-2.83,220.0,0.31,NaN,-20.43,77.32
111453,2013-12-31 20:00:00,7,9.34,NaN,1016.95,1.73,1.00,9.34,327.0,0.84,NaN,-12.93,70.58
176509,2013-12-31 20:00:00,12,9.50,NaN,1016.96,1.71,1.00,9.50,326.0,0.84,NaN,-13.03,70.67


In [20]:
irr_df = pd.read_parquet(data_path / 'external_irradiance_features.parquet')
irr_df

,site_id,hostname,g,gd,gc,a,ad,ac,td,year,month,hour
0,1,PVGIS-CMSAF,0.0,0.0,0.0,0.0,0.0,0.0,None,2013,1,6
1,1,PVGIS-CMSAF,14.0,14.0,10.0,7.0,6.0,5.0,None,2013,1,7
2,1,PVGIS-CMSAF,190.0,80.0,287.0,359.0,104.0,619.0,None,2013,1,8
3,1,PVGIS-CMSAF,331.0,118.0,524.0,490.0,133.0,844.0,None,2013,1,9
4,1,PVGIS-CMSAF,435.0,140.0,703.0,555.0,148.0,948.0,None,2013,1,10
...,...,...,...,...,...,...,...,...,...,...,...,...
8658,18,PVGIS-CMSAF,550.0,199.0,691.0,754.0,231.0,1004.0,None,2014,5,15
8659,18,PVGIS-CMSAF,395.0,170.0,466.0,696.0,221.0,930.0,None,2014,5,16
8660,18,PVGIS-CMSAF,224.0,125.0,232.0,585.0,192.0,784.0,None,2014,5,17
8661,18,PVGIS-CMSAF,81.0,70.0,57.0,317.0,113.0,419.0,None,2014,5,18


In [21]:
indexes_to_fill = df_grid.loc[irradiance_nan_indexes][['timestamp', 'site_id']]
indexes_to_fill["year"] = indexes_to_fill["timestamp"].apply(lambda x: int(x.year))
indexes_to_fill["month"] = indexes_to_fill["timestamp"].apply(lambda x: int(x.month))
indexes_to_fill["hour"] = indexes_to_fill["timestamp"].apply(lambda x: int(x.hour))
indexes_to_fill.drop(columns=["timestamp"], inplace=True)
indexes_to_fill["index"] = indexes_to_fill.index
indexes_to_fill

,site_id,year,month,hour,index
8683,1,2013,4,2,8683
8684,1,2013,4,3,8684
8685,1,2013,4,4,8685
8686,1,2013,4,5,8686
8687,1,2013,4,6,8687
...,...,...,...,...,...
268921,18,2013,4,16,268921
268922,18,2013,4,17,268922
268923,18,2013,4,18,268923
268924,18,2013,4,19,268924


In [22]:
fill_values = irr_df.merge(indexes_to_fill, on=['year', 'month', 'hour', 'site_id'], how='inner')
fill_values

,site_id,hostname,g,gd,gc,a,ad,ac,td,year,month,hour,index
0,1,PVGIS-CMSAF,33.0,32.0,18.0,16.0,13.0,9.0,None,2013,4,5,8686
1,1,PVGIS-CMSAF,95.0,72.0,83.0,226.0,84.0,341.0,None,2013,4,6,8687
2,1,PVGIS-CMSAF,255.0,130.0,317.0,539.0,176.0,847.0,None,2013,4,7,8688
3,1,PVGIS-CMSAF,411.0,171.0,571.0,630.0,201.0,981.0,None,2013,4,8,8689
4,1,PVGIS-CMSAF,542.0,194.0,796.0,675.0,211.0,1045.0,None,2013,4,9,8690
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16389,11,PVGIS-CMSAF,0.0,0.0,0.0,0.0,0.0,0.0,None,2012,12,19,153233
16390,11,PVGIS-CMSAF,0.0,0.0,0.0,0.0,0.0,0.0,None,2012,12,19,153252
16391,11,PVGIS-CMSAF,0.0,0.0,0.0,0.0,0.0,0.0,None,2012,12,19,153271
16392,11,PVGIS-CMSAF,0.0,0.0,0.0,0.0,0.0,0.0,None,2012,12,19,153290


In [23]:
# show non filled rows in indexes_to_fill that are not in fill_values
non_filled_rows = indexes_to_fill.merge(
    fill_values, 
    on=['year', 'month', 'hour', 'site_id'],
    how='left', indicator=True
).query('_merge == "left_only"').drop(columns=['_merge'])
print("Number of non filled rows for each year, month and hour:")
for col in ['year', 'month', 'hour']:
    print(f"{col} value counts:")
    print(non_filled_rows[col].value_counts().to_frame().T)
non_filled_rows

Number of non filled rows for each year, month and hour:
year value counts:
year   2013  2012
count  4170   722
month value counts:
month    12    11   10   9    8    7   6   4   5 
count  1147  1030  820  720  577  465  77  32  24
hour value counts:
hour     2     3     4    20   5 
count  1114  1114  1086  923  655


,site_id,year,month,hour,index_x,hostname,g,gd,gc,a,ad,ac,td,index_y
0,1,2013,4,2,8683,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2013,4,3,8684,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,2013,4,4,8685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,1,2013,4,20,8701,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,2,2013,4,2,24947,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
476949,17,2013,12,20,257829,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
476950,18,2013,4,2,268907,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
476951,18,2013,4,3,268908,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
476952,18,2013,4,4,268909,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# fill df_grid with fill_values based on year, month, hour, site_id
# then for any remaining NaN values in df_grid["irradiance"], 
# fill them with 0 irradiance since all site_id, month, and hour combination 
# not present in the irradiance data are late-night, early-morning, or winter-evening hours in Italy
# where there is no sunlight and the irradiance is expected to be 0
indexes_to_fill = fill_values["index"]
df_grid.loc[indexes_to_fill, "irradiance"] = fill_values["a"].values

In [25]:
df_grid["irradiance"] = df_grid["irradiance"].fillna(0)

## Impute remaining `kwh` NaN values

In [26]:
# --- STEP 3: HIERARCHICAL CLIMATOLOGICAL IMPUTATION ---
# Extract helper temporal features
df_grid['year'] = df_grid['timestamp'].dt.year
df_grid['month'] = df_grid['timestamp'].dt.month
df_grid['hour'] = df_grid['timestamp'].dt.hour
    
# 2. Weather Features Fallback: Impute any remaining NaNs using Site-Month-Hour averages
grp_month_hour = ['site_id', 'month', 'hour']

# 3. Target Feature (kWh): Impute using Site-Month-Hour averages (as per paper Section A)
target_climatology = df_grid.groupby(grp_month_hour)["kwh"].transform('mean')

kwh_nan_indexes = df_grid[df_grid["kwh"].isnull()].index

df_grid["kwh"] = df_grid["kwh"].fillna(target_climatology)

# Clean up temporal helpers and return
df_grid = df_grid.drop(columns=['year', 'month', 'hour'])


# Visualize the data

In [27]:
# show any row with NaN values in df_grid
df_grid[df_grid.isnull().any(axis=1)]

,timestamp,site_id,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh,altitude,azimuth


In [28]:
df_grid

,timestamp,site_id,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh,altitude,azimuth
0,2012-01-02 02:00:00,1,0.0,0.0,0.00,2.24,0.82,-1.34,260.0,0.00,0.0,-71.88,-171.68
1,2012-01-02 03:00:00,1,0.0,0.0,1025.41,0.84,0.85,-1.46,206.0,0.11,0.0,-69.31,-146.58
2,2012-01-02 04:00:00,1,0.0,0.0,1025.41,3.58,0.84,-1.69,210.0,0.11,0.0,-60.96,-119.88
3,2012-01-02 05:00:00,1,0.0,0.0,1025.41,3.58,0.70,-1.08,220.0,0.11,0.0,-50.42,-103.84
4,2012-01-02 06:00:00,1,0.0,5.0,1025.16,3.58,0.82,-1.77,230.0,0.11,10.0,-39.21,-92.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...
276483,2014-05-06 16:00:00,18,19.0,123.0,1020.20,1.49,0.56,9.41,76.0,0.31,25.0,55.23,54.37
276484,2014-05-06 17:00:00,18,18.0,23.0,1020.20,1.93,0.58,9.28,62.0,0.31,3.0,45.18,70.55
276485,2014-05-06 18:00:00,18,17.0,1.0,1019.91,0.89,0.63,8.82,89.0,0.31,0.0,34.17,82.61
276486,2014-05-06 19:00:00,18,16.0,1.0,1019.91,0.95,0.73,8.77,132.0,0.31,0.0,22.87,92.73


In [29]:
# apply minmax scaling to the features in df_grid
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
dropped_df = df_grid.drop(columns=["timestamp", "site_id"])
df_grid_scaled = scaler.fit_transform(dropped_df)
df_grid_scaled = pd.DataFrame(df_grid_scaled, columns=dropped_df.columns)
df_grid_scaled[["timestamp", "site_id"]] = df_grid[["timestamp", "site_id"]]
df_grid_scaled

,temperature,irradiance,pressure,windspeed,humidity,dewpoint,windbearing,cloudcover,kwh,altitude,azimuth,timestamp,site_id
0,0.215686,0.000000,0.000000,0.070774,0.800000,0.435497,0.724234,0.00,0.000,0.008150,0.028520,2012-01-02 02:00:00,1
1,0.215686,0.000000,0.990275,0.026540,0.833333,0.433122,0.573816,0.11,0.000,0.025750,0.114558,2012-01-02 03:00:00,1
2,0.215686,0.000000,0.990275,0.113112,0.822222,0.428571,0.584958,0.11,0.000,0.082934,0.206081,2012-01-02 04:00:00,1
3,0.215686,0.000000,0.990275,0.113112,0.666667,0.440641,0.612813,0.11,0.000,0.155116,0.261063,2012-01-02 05:00:00,1
4,0.215686,0.003021,0.990034,0.113112,0.800000,0.426989,0.640669,0.11,0.010,0.231886,0.300278,2012-01-02 06:00:00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
276483,0.588235,0.074320,0.985244,0.047077,0.511111,0.648199,0.211699,0.31,0.025,0.878647,0.803380,2014-05-06 16:00:00,18
276484,0.568627,0.013897,0.985244,0.060979,0.533333,0.645627,0.172702,0.31,0.003,0.809821,0.858842,2014-05-06 17:00:00,18
276485,0.549020,0.000604,0.984963,0.028120,0.588889,0.636526,0.247911,0.31,0.000,0.734420,0.900182,2014-05-06 18:00:00,18
276486,0.529412,0.000604,0.984963,0.030016,0.700000,0.635536,0.367688,0.31,0.000,0.657033,0.934871,2014-05-06 19:00:00,18


In [30]:
import os

from plots import plot_data

nan_indexes_from_feat = {
    "azimuth": az_alt_nan_indexes,
    "altitude": az_alt_nan_indexes,
    
    "irradiance": irradiance_nan_indexes,

    "kwh": kwh_nan_indexes,

    # ambiental_nan_indexes
    "temperature": ambiental_nan_indexes,
    "pressure": ambiental_nan_indexes,
    "windspeed": ambiental_nan_indexes,
    "humidity": ambiental_nan_indexes,
    "dewpoint": ambiental_nan_indexes,
    "windbearing": ambiental_nan_indexes,
    "cloudcover": ambiental_nan_indexes,
    
}

# for node in [8]:
for node in range(1, 18):
    # for feature in ["azimuth", "altitude"]:
    for feature in dropped_df.columns:
        fig = plot_data(df_grid, feature, node, nan_indexes_from_feat.get(feature))
        # fig.show()
        os.makedirs(f"./plots/pv-italy/reversed/raw/{node}", exist_ok=True)
        fig.write_html(f"./plots/pv-italy/reversed/raw/{node}/{feature}.html")